## PGA – GradCAM Rescue + IPR

**Yêu cầu**: Đã train model PGA-UNet và lưu checkpoint lên Google Drive.

Chạy tuần tự 3 cells: Setup → Download checkpoint → Rescue Pipeline

In [ ]:
# =========================================================
# CELL 1 - SETUP: Clone repo + Download dataset
# =========================================================
%cd /content

# Clone repo (branch TN_B_ON)
!git clone -b TN_B_ON https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git

# Download dataset từ Google Drive
!gdown --id 1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW

# Giải nén + di chuyển vào project
!unzip -oq dataset_BTXRD.zip
!mv dataset_BTXRD Prompt-Guided-XRay-Segmentation/

# Vào thư mục project
%cd Prompt-Guided-XRay-Segmentation

# Cài dependencies
!pip install -q tqdm opencv-python matplotlib scikit-image gdown scipy

print("\n✅ Setup done!")


## Bước 1 – Tải checkpoint từ Google Drive

In [ ]:
import gdown, os

CKPT_FILE_ID = "1Mv-rUPI7KGmYemd27hmKbJQRHc4ZKB9z"   # trọng số cũ — kết quả GradCAM tốt hơn

SAVE_PATH = "checkpoints/pga_unet_expB_best.pth"
os.makedirs("checkpoints", exist_ok=True)

url = f"https://drive.google.com/uc?id={CKPT_FILE_ID}"
gdown.download(url, SAVE_PATH, quiet=False)

import os.path
assert os.path.exists(SAVE_PATH), f"❌ Tải thất bại: {SAVE_PATH}"
print(f"✅ Checkpoint sẵn sàng: {SAVE_PATH}  ({os.path.getsize(SAVE_PATH)//1024} KB)")

## Thí Nghiệm B — GradCAM Rescue + IPR

Chạy tuần tự 3 cell:
1. **Phòng hộ** — load model, kiểm tra Achievement 1, hiện cá lọt lưới
2. **Số liệu** — GradCAM CBL stats + bảng IPR convergence (6 metrics)
3. **Ảnh** — trực quan hóa rescue samples + phân bin

In [ ]:
# @title  Cell 1 — Phòng hộ vùng đen  (predict_with_check, giống file cũ)
import os, cv2, torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm import tqdm

from dataset import BTXRD_Dataset
from models.networks.prompt_unet_2D import PGA_UNet

# ── Config ────────────────────────────────────────────────────────────
DEVICE               = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH           = "checkpoints/pga_unet_expB_best.pth"
IMG_SIZE             = 512
USE_ENCODER_PROMPT   = True
CONFIDENCE_THRESHOLD = 0.80
CENTER_DIST_RATIO    = 0.25
DARK_PIXEL_THRESHOLD = -0.80
DARK_RATIO_LIMIT     = 0.70
TEST_IMAGE_DIR = "dataset_BTXRD/test/images"
TEST_JSON_DIR  = "dataset_BTXRD/test/annotations"

# ── Helpers dùng chung cho cả 3 cell ─────────────────────────────────
def extract_lcc(m):
    if m.sum() == 0: return m
    n, lbl, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), 8)
    if n <= 1: return m
    return (lbl == (1 + np.argmax(st[1:, cv2.CC_STAT_AREA]))).astype(np.float32)

def get_centroid(pm):
    if pm.sum() == 0: return None, None
    ys, xs = np.where(pm > 0.5)
    return float(xs.mean()), float(ys.mean())

def compute_gradcam(model, image_tensor):
    grads, acts = [], []
    def _hook(m, i, o):
        acts.append(o)
        o.register_hook(lambda g: grads.append(g))
    h = model.center.register_forward_hook(_hook)
    model.eval()
    img_t = image_tensor.clone().detach().to(DEVICE)
    zero_p = torch.zeros(1, 1, IMG_SIZE, IMG_SIZE, device=DEVICE)
    out = model(img_t, zero_p)
    model.zero_grad(); out.sum().backward(); h.remove()
    if not grads: return None
    w   = grads[0].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * acts[0]).sum(1, keepdim=True))
    cam = F.interpolate(cam, (IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
    cam = cam[0, 0].detach().cpu().numpy()
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

def predict_with_check(model, img_tensor, prompt_tensor):
    """Copy từ file cũ — 5 điều kiện nghi ngờ, kích hoạt GradCAM nếu suspicious."""
    model.eval()
    with torch.no_grad():
        out  = model(img_tensor.to(DEVICE), prompt_tensor.to(DEVICE))
        prob = torch.sigmoid(out)
        pred = (prob > 0.5).float()
    conf    = prob.max().item()
    pm_np   = prompt_tensor[0, 0].numpy()
    pr_np   = pred[0, 0].cpu().numpy()
    img_np  = img_tensor[0, 0].numpy()
    prob_np = prob[0, 0].cpu().numpy()
    ys_p, xs_p = np.where(pm_np > 0.3)
    cx_pmt = xs_p.mean() if len(xs_p) else IMG_SIZE / 2
    cy_pmt = ys_p.mean() if len(ys_p) else IMG_SIZE / 2
    prompt_area = len(xs_p)
    cx_pr, cy_pr = get_centroid(pr_np)
    pred_area    = int(pr_np.sum())
    dist = (np.sqrt((cx_pr - cx_pmt)**2 + (cy_pr - cy_pmt)**2)
            if cx_pr is not None else IMG_SIZE)
    s_conf  = conf < CONFIDENCE_THRESHOLD
    s_dist  = dist > IMG_SIZE * CENTER_DIST_RATIO
    s_area  = pred_area < 50
    s_ratio = prompt_area > 0 and (pred_area / float(prompt_area)) < 0.05
    pm_mask = pm_np > 0.3
    if pm_mask.sum() > 0:
        dark_ratio = float((img_np[pm_mask] < DARK_PIXEL_THRESHOLD).sum() / pm_mask.sum())
        s_dark = dark_ratio > DARK_RATIO_LIMIT
    else:
        dark_ratio = 1.0; s_dark = True
    is_susp = any([s_conf, s_dist, s_area, s_ratio, s_dark])
    if is_susp:
        pr_np = np.zeros_like(pr_np); cx_pr = cy_pr = None
        sal = compute_gradcam(model, img_tensor)
    else:
        sal = None
    return dict(mask=pr_np, prob_map=prob_np, confidence=conf,
                cx_pred=cx_pr, cy_pred=cy_pr, is_suspicious=is_susp,
                saliency=sal, dark_ratio=dark_ratio,
                reasons=dict(CONF=s_conf, DIST=s_dist, AREA=s_area, RATIO=s_ratio, DARK=s_dark))

# ── Load model ────────────────────────────────────────────────────────
model = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=USE_ENCODER_PROMPT).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.eval()
print(f"✅ Loaded: {MODEL_PATH}")

test_ds = BTXRD_Dataset(TEST_IMAGE_DIR, TEST_JSON_DIR,
                        img_size=IMG_SIZE, is_train=False, prompt_mode='zoom_out')
loader  = DataLoader(test_ds, batch_size=1, shuffle=False)

# ── Quét phòng hộ (6 góc, 70%) ───────────────────────────────────────
BOX = 80
CORNERS = [(10, 10), (IMG_SIZE-BOX-10, 10),
           (10, IMG_SIZE-BOX-10), (IMG_SIZE-BOX-10, IMG_SIZE-BOX-10),
           (IMG_SIZE//2, 10), (10, IMG_SIZE//2)]

n_dark = 0; n_rescued = 0
rescued_items = []   # samples bị bắt (is_suspicious=True) → đưa vào Cell 2
failed_items  = []   # cá lọt lưới (is_suspicious=False)

for img_t, mask_t, _ in tqdm(loader, desc="Phòng hộ"):
    img_np = img_t[0, 0].numpy()
    gt_np  = mask_t[0, 0].numpy()
    if gt_np.sum() == 0: continue

    # Tìm góc tối nhất
    best_ratio, best_box = -1.0, (10, 10, 10+BOX, 10+BOX)
    for x, y in CORNERS:
        r = float((img_np[y:y+BOX, x:x+BOX] < DARK_PIXEL_THRESHOLD).mean())
        if r > best_ratio: best_ratio = r; best_box = (x, y, x+BOX, y+BOX)
    if best_ratio < DARK_RATIO_LIMIT: continue
    n_dark += 1

    x1, y1, x2, y2 = best_box
    dark_hm = test_ds.create_plateau_heatmap([x1, y1, x2, y2], IMG_SIZE, IMG_SIZE)
    pm_t    = torch.from_numpy(dark_hm).unsqueeze(0).unsqueeze(0)

    res = predict_with_check(model, img_t, pm_t)

    item = dict(img=img_np, gt=gt_np, img_t=img_t,
                dark_box=best_box, dark_ratio=best_ratio,
                dark_hm=dark_hm, res=res)
    if res['is_suspicious']:
        n_rescued += 1
        rescued_items.append(item)
    else:
        failed_items.append(item)

# ── Báo cáo ──────────────────────────────────────────────────────────
bar = "═" * 58
print(f"\n{bar}\n  PHÒNG HỘ VÙNG TỐI\n{bar}")
print(f"  Mẫu có góc tối ≥70%                 : {n_dark}")
print(f"  Bắt thành công (is_suspicious=True)  : {n_rescued}  ← tập test GradCAM+IPR")
print(f"  Cá lọt lưới   (is_suspicious=False)  : {len(failed_items)}")
rate = n_rescued / n_dark * 100 if n_dark else 0
tok  = "✅" if rate >= 90 else "⚠️ "
print(f"  {tok} Tỷ lệ phòng hộ: {n_rescued}/{n_dark} = {rate:.1f}%")
print(bar)

# ── Hiện cá lọt lưới ─────────────────────────────────────────────────
if failed_items:
    print(f"\n⚠️  {len(failed_items)} cá lọt lưới:")
    H = W = IMG_SIZE
    for i, item in enumerate(failed_items):
        res   = item['res']
        imgd  = np.clip(item['img'] * 0.5 + 0.5, 0, 1)
        gt    = item['gt']
        print(f"\n  Cá #{i+1}: corner_dark={item['dark_ratio']:.3f}  "
              f"prompt_dark={res['dark_ratio']:.3f}  conf={res['confidence']:.3f}")
        print(f"    Lý do KHÔNG bị bắt: {[k for k,v in res['reasons'].items() if not v]}")
        fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
        fig.suptitle(f"Cá lọt lưới #{i+1}  |  corner_dark={item['dark_ratio']:.3f}  "
                     f"conf={res['confidence']:.3f}  prompt_dark={res['dark_ratio']:.3f}",
                     fontsize=9, color='darkred')
        axes[0].imshow(imgd, cmap='gray'); axes[0].set_title("Ảnh gốc", fontsize=8)
        axes[1].imshow(imgd, cmap='gray')
        dh = item['dark_hm'].astype(np.float32)
        axes[1].imshow(np.ma.masked_where(dh < 0.1, dh), cmap='Reds', alpha=0.4)
        x1, y1, x2, y2 = item['dark_box']
        axes[1].add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                        lw=2, edgecolor='red', facecolor='none'))
        ov = np.zeros((H, W, 4)); ov[res['prob_map'] > 0.5] = [1, .2, .2, .7]
        axes[1].imshow(ov)
        axes[1].set_title(f"Prob map (dark)\nconf={res['confidence']:.3f}", fontsize=8)
        axes[2].imshow(imgd, cmap='gray')
        ov = np.zeros((H, W, 4)); ov[res['mask'] > 0] = [.1, .9, .1, .6]
        axes[2].imshow(ov)
        axes[2].set_title(f"Mask ra (lọt!)\npx={int(res['mask'].sum())}", fontsize=8, color='red')
        axes[3].imshow(imgd, cmap='gray')
        ov = np.zeros((H, W, 4)); ov[gt > 0.5] = [.1, .4, 1, .6]
        axes[3].imshow(ov); axes[3].set_title("GT", fontsize=8)
        for ax in axes: ax.axis('off')
        plt.tight_layout(); plt.show()
else:
    print("\n✅ Không có cá lọt lưới!")

print(f"\n→ Chạy Cell 2 để test GradCAM+IPR trên {n_rescued} mẫu bị bắt.")

In [ ]:
# @title  Cell 2 — GradCAM + IPR: Số liệu
# Chạy sau Cell 1.  N = len(rescued_items) — chỉ tính trên tập bị bắt phòng hộ.
from scipy.ndimage import binary_erosion, distance_transform_edt

NUM_IPR_STEPS   = 5
NUM_VIS_RESCUE  = 10
NUM_VIS_PER_BIN = 3

BINS = [
    ("0.0 – 0.5",  0.0, 0.50),
    ("0.5 – 0.7",  0.5, 0.70),
    ("0.7 – 0.9",  0.7, 0.90),
    ("0.9 – 0.99", 0.9, 1.00),
]

# ── Metrics ───────────────────────────────────────────────────────────
def calc_hd95(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or  not g.any(): return float(IMG_SIZE)
    pe = p ^ binary_erosion(p); ge = g ^ binary_erosion(g)
    d1 = distance_transform_edt(~ge)[pe]; d2 = distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return float(IMG_SIZE)
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_all_metrics(prob, gm):
    pm  = extract_lcc((prob > 0.5).astype(np.float32))
    gm2 = (gm > 0.5).astype(np.float32)
    eps = 1e-6
    tp = (pm*gm2).sum(); fp = (pm*(1-gm2)).sum(); fn = ((1-pm)*gm2).sum()
    dice = (2*tp+eps)/(2*tp+fp+fn+eps); iou  = (tp+eps)/(tp+fp+fn+eps)
    prec = (tp+eps)/(tp+fp+eps);        rec  = (tp+eps)/(tp+fn+eps)
    hd95 = calc_hd95(pm, gm2)
    if gm2.sum() == 0 or pm.sum() == 0:
        cbl = 0.0
    else:
        ys,xs = np.where(gm2>0.5); yp,xp = np.where(pm>0.5)
        gt_d  = np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl   = float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/gt_d,0,1))
    return dict(mask=pm, dice=float(dice), iou=float(iou),
                precision=float(prec), recall=float(rec), hd95=hd95, cbl=cbl)

def calc_cbl_point(px, py, gt_bin):
    if gt_bin.sum() == 0: return None
    ys, xs = np.where(gt_bin > 0.5)
    gt_d = np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+1e-6
    return float(np.clip(1.-np.sqrt((px-xs.mean())**2+(py-ys.mean())**2)/gt_d, 0, 1))

# ── GradCAM + IPR trên tập rescued_items ─────────────────────────────
if not rescued_items:
    print("⚠️  rescued_items rỗng — chạy Cell 1 trước."); raise SystemExit

r = (test_ds.zoom_ratio[0] + test_ds.zoom_ratio[1]) / 2
gcam_cbl_list = []; per_sample = []

for item in tqdm(rescued_items, desc="GradCAM + IPR"):
    img_np = item['img']; gt_np = item['gt']
    res    = item['res']

    sal = res['saliency']   # đã tính trong predict_with_check
    if sal is None: continue

    img_dev = item['img_t'].to(DEVICE)

    # GradCAM CBL
    py_c, px_c = np.unravel_index(sal.argmax(), sal.shape)
    cbl_peak   = calc_cbl_point(float(px_c), float(py_c), gt_np)
    if cbl_peak is not None: gcam_cbl_list.append(cbl_peak)

    ys_g, xs_g = np.where(gt_np > 0.5)
    bw = (xs_g.max()-xs_g.min()) * (1+r)
    bh = (ys_g.max()-ys_g.min()) * (1+r)
    px_curr, py_curr = float(px_c), float(py_c)

    steps = []
    for k in range(NUM_IPR_STEPS):
        pm_map = test_ds.create_plateau_heatmap(
            [px_curr-bw/2, py_curr-bh/2, px_curr+bw/2, py_curr+bh/2],
            IMG_SIZE, IMG_SIZE)
        pm_t = torch.from_numpy(cv2.resize(pm_map, (IMG_SIZE, IMG_SIZE))).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            prob = torch.sigmoid(model(img_dev, pm_t))[0, 0].cpu().numpy()
        m  = calc_all_metrics(prob, gt_np)
        cx, cy = get_centroid(m['mask'])
        if cx is not None: px_curr, py_curr = cx, cy
        steps.append({**m, 'pm': pm_map, 'cx': cx, 'cy': cy, 'k': k+1})

    per_sample.append(dict(
        img       = img_np,
        gt        = gt_np,
        sal       = sal,
        dark_box  = item['dark_box'],
        dark_ratio= item['dark_ratio'],          # corner dark ratio
        dark_hm   = item['dark_hm'],
        prob_map_dark = res['prob_map'],          # prob với dark prompt
        gcam_cbl  = cbl_peak if cbl_peak is not None else 0.0,
        gcam_dice = steps[0]['dice'] if steps else 0.0,
        steps     = steps,
        final     = steps[-1] if steps else {},
    ))

N = len(per_sample)
print(f"\n✅ IPR hoàn thành trên {N} mẫu (tập bị phòng hộ)")

# ── GradCAM CBL stats ─────────────────────────────────────────────────
if gcam_cbl_list:
    a   = np.array(gcam_cbl_list); tot = len(a)
    g1  = (a < 0.5).sum(); g2 = ((a>=0.5)&(a<0.7)).sum()
    g3  = ((a>=0.7)&(a<0.9)).sum(); g4 = (a>=0.9).sum()
    bar = "═"*55
    print(f"\n{bar}\n  GRADCAM PEAK CBL  (N={tot})\n{bar}")
    print(f"  Mean={a.mean():.4f}  Max={a.max():.4f}  Min={a.min():.4f}")
    for label, cnt in [("[0.0–0.5)  Yếu", g1),("[0.5–0.7)  Trung bình", g2),
                        ("[0.7–0.9)  Tốt", g3),("[0.9–1.0]  Xuất sắc", g4)]:
        print(f"  {label:<22} {cnt:3d} ({cnt/tot*100:5.1f}%)")

# ── Bảng IPR ─────────────────────────────────────────────────────────
KEYS = ["dice","iou","precision","recall","hd95","cbl"]
HDRS = ["Dice↑","IoU↑","Prec↑","Rec↑","HD95↓","CBL↑"]

def _bin_groups():
    groups = {b[0]:[] for b in BINS}
    for s in per_sample:
        d = s['gcam_dice']
        for bn,lo,hi in BINS:
            if lo <= d < hi: groups[bn].append(s); break
    return groups

def _row(samples, si):
    if not samples: return {k: 0. for k in KEYS}
    return {k: np.mean([s['steps'][si][k] for s in samples if si < len(s['steps'])]) for k in KEYS}

groups = _bin_groups()
bar = "═"*78
for si, label in [(0, "BẢNG 1 — IPR k=1  (GradCAM → prompt đầu)"),
                  (NUM_IPR_STEPS-1, f"BẢNG 2 — IPR k={NUM_IPR_STEPS} (vòng cuối)")]:
    print(f"\n{bar}\n  {label}  |  N={N}\n{bar}")
    print(f"  {'Nhóm':<13}{'N':>4}  "+"".join(f"{h:>9}" for h in HDRS))
    print(f"  {'─'*74}")
    for bn,lo,hi in BINS:
        g = groups[bn]; m = _row(g, si)
        print(f"  {bn:<13}{len(g):>4}  "+"".join(f"{m[k]:>9.2f}" if k=='hd95' else f"{m[k]:>9.4f}" for k in KEYS))
    print(f"  {'─'*74}")
    ov = {k: np.mean([s['steps'][si][k] for s in per_sample if si < len(s['steps'])]) for k in KEYS}
    print(f"  {'Tổng/Mean':<13}{N:>4}  "+"".join(f"{ov[k]:>9.2f}" if k=='hd95' else f"{ov[k]:>9.4f}" for k in KEYS))
    print(bar)

print(f"\n{bar}\n  IPR CONVERGENCE  (N={N})\n{bar}")
print(f"  {'Bước':<22}"+"".join(f"{h:>9}" for h in ["Dice↑","IoU↑","HD95↓","CBL↑"]))
print(f"  {'─'*58}")
for v in range(NUM_IPR_STEPS):
    ks   = ["dice","iou","hd95","cbl"]
    vals = {k: np.mean([s['steps'][v][k] for s in per_sample if v < len(s['steps'])]) for k in ks}
    print(f"  {f'IPR k={v+1}':<22}"+"".join(f"{vals[k]:>9.2f}" if k=='hd95' else f"{vals[k]:>9.4f}" for k in ks))
print(bar)

In [ ]:
# @title  Cell 3 — Show ảnh
# Chạy sau Cell 2. Columns: Ảnh+Dark | GradCAM | RescueBox | IPR k=1..5 | GT
import matplotlib.patches as mpatches

if not per_sample:
    print("⚠️  per_sample rỗng — chạy Cell 2 trước."); raise SystemExit

H = W = IMG_SIZE
# Cột: 0=Ảnh+dark  1=GradCAM  2=RescueBox  3..7=IPR k=1..5  8=GT
N_COLS  = 3 + NUM_IPR_STEPS + 1
IPR_LBLS = [f"IPR k={k}" for k in range(1, NUM_IPR_STEPS+1)]
TITLES   = ["Ảnh+Dark Prompt", "GradCAM\n(Rescue hint)", "Rescue Box\n(k=1 init)"] \
           + IPR_LBLS + ["GT"]

# ── Showcase ─────────────────────────────────────────────────────────
samples = per_sample[:NUM_VIS_RESCUE]
n = len(samples)
fig, axes = plt.subplots(n, N_COLS, figsize=(N_COLS*2.8, n*3.0), squeeze=False)
fig.suptitle(f"GradCAM Rescue — {n} mẫu  |  {NUM_IPR_STEPS} IPR steps  |  N_total={len(per_sample)}",
             fontsize=11, fontweight='bold', y=1.01)
for j, t in enumerate(TITLES):
    axes[0, j].set_title(t, fontsize=7, fontweight='bold')

for ri, s in enumerate(samples):
    imgd = np.clip(s['img'].astype(np.float32)*0.5+0.5, 0, 1)

    # Col 0 — Ảnh + dark box
    ax = axes[ri, 0]; ax.imshow(imgd, cmap='gray', vmin=0, vmax=1)
    dh = s['dark_hm'].astype(np.float32)
    ax.imshow(np.ma.masked_where(dh < 0.1, dh), cmap='Reds', alpha=0.4)
    x1, y1, x2, y2 = s['dark_box']
    ax.add_patch(plt.Rectangle((x1,y1), x2-x1, y2-y1, lw=2, edgecolor='red', facecolor='none'))
    ax.set_xlabel(f"dark={s['dark_ratio']:.2f}", fontsize=6, color='red')
    ax.axis('off')

    # Col 1 — GradCAM
    ax = axes[ri, 1]; ax.imshow(imgd, cmap='gray', vmin=0, vmax=1)
    ax.imshow(s['sal'].astype(np.float32), cmap='jet', alpha=0.5)
    py_c, px_c = np.unravel_index(s['sal'].argmax(), s['sal'].shape)
    ax.plot(px_c, py_c, '*', color='yellow', ms=8, markeredgecolor='black', markeredgewidth=0.5)
    ax.set_xlabel(f"CBL={s['gcam_cbl']:.3f}", fontsize=6, color='darkorange')
    ax.axis('off')

    # Col 2 — Rescue Box (prompt của IPR k=1)
    ax = axes[ri, 2]; ax.imshow(imgd, cmap='gray', vmin=0, vmax=1)
    if s['steps']:
        pm0 = s['steps'][0]['pm'].astype(np.float32)
        ax.imshow(np.ma.masked_where(pm0 < 0.1, pm0), cmap='magma', alpha=0.55)
    ax.axis('off')

    # Col 3..7 — IPR k=1..5
    for v in range(NUM_IPR_STEPS):
        ax = axes[ri, 3+v]; ax.imshow(imgd, cmap='gray', vmin=0, vmax=1)
        if v < len(s['steps']):
            st   = s['steps'][v]
            mask = st['mask'].astype(np.float32)
            g_c  = float(np.clip(0.4 + 0.6*st['dice'], 0, 1))
            ov   = np.zeros((H, W, 4), dtype=np.float32)
            ov[mask > 0.5] = [0.05, g_c, 0.1, 0.75]
            ax.imshow(ov)
            if st['cx'] is not None:
                ax.plot(float(st['cx']), float(st['cy']), 'o', color='yellow', ms=3)
            clr = 'darkgreen' if st['dice'] > 0.5 else 'orangered'
            ax.set_xlabel(f"D={st['dice']:.3f}", fontsize=6, color=clr)
        ax.axis('off')

    # Col cuối — GT
    ax = axes[ri, N_COLS-1]; ax.imshow(imgd, cmap='gray', vmin=0, vmax=1)
    ov = np.zeros((H, W, 4), dtype=np.float32)
    ov[s['gt'] > 0.5] = [0.1, 0.4, 1.0, 0.65]
    ax.imshow(ov); ax.axis('off')

plt.tight_layout()
plt.savefig("gradcam_rescue_showcase.png", dpi=100, bbox_inches='tight')
plt.show()
print(f"✅ gradcam_rescue_showcase.png  ({N_COLS} cột × {n} hàng)")

# ── Bin analysis ──────────────────────────────────────────────────────
print(f"\n{'─'*58}\n  PHÂN NHÓM THEO DICE IPR k=1\n{'─'*58}")
for bn, lo, hi in BINS:
    g  = groups[bn]
    md = np.mean([s['steps'][0]['dice'] for s in g]) if g else 0.
    print(f"  [{bn}] : {len(g):3d} mẫu  |  mean Dice k=1 = {md:.4f}")
print(f"{'─'*58}")

for bn, lo, hi in BINS:
    g = groups[bn]
    if not g: continue
    g_sort = sorted(g, key=lambda s: s['gcam_dice'])
    idxs   = np.linspace(0, len(g_sort)-1, min(NUM_VIS_PER_BIN, len(g_sort)), dtype=int)
    chosen = [g_sort[i] for i in idxs]
    nr     = len(chosen)

    fig, axes = plt.subplots(nr, N_COLS, figsize=(N_COLS*2.8, nr*3.0), squeeze=False)
    fig.suptitle(f"Cụm Dice IPR-k1 {bn}  —  {nr}/{len(g)} mẫu", fontsize=11, fontweight='bold')
    for j, t in enumerate(TITLES): axes[0, j].set_title(t, fontsize=7, fontweight='bold')

    for ri, s in enumerate(chosen):
        imgd = np.clip(s['img'].astype(np.float32)*0.5+0.5, 0, 1)

        # Col 0 — Ảnh + dark
        ax = axes[ri, 0]; ax.imshow(imgd, cmap='gray', vmin=0, vmax=1)
        dh = s['dark_hm'].astype(np.float32)
        ax.imshow(np.ma.masked_where(dh<0.1,dh), cmap='Reds', alpha=0.35)
        x1,y1,x2,y2 = s['dark_box']
        ax.add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1,lw=2,edgecolor='red',facecolor='none'))
        ax.set_xlabel(f"dark={s['dark_ratio']:.2f}", fontsize=6, color='red')
        ax.axis('off')

        # Col 1 — GradCAM
        ax = axes[ri, 1]; ax.imshow(imgd, cmap='gray', vmin=0, vmax=1)
        ax.imshow(s['sal'].astype(np.float32), cmap='jet', alpha=0.45)
        py_c, px_c = np.unravel_index(s['sal'].argmax(), s['sal'].shape)
        ax.plot(px_c, py_c, '*', color='yellow', ms=8, markeredgecolor='black', markeredgewidth=0.5)
        ax.set_xlabel(f"CBL={s['gcam_cbl']:.3f}", fontsize=6, color='darkorange')
        ax.axis('off')

        # Col 2 — Rescue Box
        ax = axes[ri, 2]; ax.imshow(imgd, cmap='gray', vmin=0, vmax=1)
        if s['steps']:
            pm0 = s['steps'][0]['pm'].astype(np.float32)
            ax.imshow(np.ma.masked_where(pm0<0.1,pm0), cmap='magma', alpha=0.5)
        ax.axis('off')

        # Col 3..7 — IPR k=1..5
        for v in range(NUM_IPR_STEPS):
            ax = axes[ri, 3+v]; ax.imshow(imgd, cmap='gray', vmin=0, vmax=1)
            if v < len(s['steps']):
                st   = s['steps'][v]
                mask = st['mask'].astype(np.float32)
                ov   = np.zeros((H, W, 4), dtype=np.float32)
                ov[mask > 0.5] = [0.1, 0.9, 0.1, 0.55]
                ax.imshow(ov)
                if st['cx'] is not None:
                    ax.plot(float(st['cx']), float(st['cy']), 'o', color='white', ms=3)
                ax.set_xlabel(f"D={st['dice']:.3f}  CBL={st['cbl']:.3f}",
                              fontsize=5.5, color='darkgreen' if st['dice']>0.5 else 'orangered')
            ax.axis('off')

        # Col cuối — GT
        ax = axes[ri, N_COLS-1]; ax.imshow(imgd, cmap='gray', vmin=0, vmax=1)
        ov = np.zeros((H, W, 4), dtype=np.float32)
        ov[s['gt'] > 0.5] = [0.1, 0.4, 1.0, 0.6]
        ax.imshow(ov); ax.axis('off')

    plt.tight_layout()
    fname = f"bin_{bn.replace(' ','').replace('–','_').replace('.','')}.png"
    plt.savefig(fname, dpi=100, bbox_inches='tight')
    plt.show()
    print(f"✅ {fname}")